# Labelled examples

This exploratory notebook serves for the creation of labelled examples

In [3]:
import pandas as pd
import json
from collections import Counter
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import re
import pycountry
from src.text_processing_functions import *
from src.LLM_functions import *
import copy as cp

from src.data import *



In [19]:
#Load data
file_path = DATA_IN_PATH +'filtered_report_types_nat_hazards_summary-header.json'#'nathaz_ifrc_reports_info_processed.json'

# Open and read the JSON file
with open(file_path, 'r') as json_file:
    filtered_reports = json.load(json_file)
filtered_reports = pd.DataFrame(filtered_reports)

## Report labelling

Choose a hazard directory. In the case below we use : \
hazard_all_subtype_emdat = {
“drought”, 
“forest fire”, “land fire”, 
“ground movement”, “tsunami”, 
“avalanche”, “landslide”, “rockfall”, “sudden subsidence”, “mudslide", 
“ash fall”, “lava flow”, “pyroclastic flow”, “lahar”, 
“coastal flood”, “flash flood”, “riverine flood”, “ice jam flood”,
“rogue wave”, “seiche”, 
”coldwave”, “heatwave”, “severe winter conditions”, 
“derecho”, “hail”, “lightning/thunderstorm”, “sand/dust storm”,  “winter storm/blizzard”, “storm surge”, “tornado”, “extra-tropical storm”, “tropical cyclone”
}

The flood subtype being hard to differentiate, we will assign hazard to "flash flood" when the text mention heavy rain, "riverine flood" if nothing specific is mentioned. 


Choose a hazard dict : 
hazard_subtype_emdat = {
'Drought': r"drought.", 

'Wildfire': r"wildfire.|forest fire.|land fire." , 

‘Earthquake’ : r”ground movement.|tsunami.”, 

‘Mass movement’: r"avalanche.|landslide.|rockfall.|sudden subsidence.|mudslide.",

‘Volcanic activity’ : r“ash fall.|lava flow.|pyroclastic flow.|lahar”, 

'Flood': r"(coastal flood.|flash flood.|riverine flood.|ice jam flood.)",

‘Wave action’ : r“rogue wave.|seiche”,

‘Extreme temperature’ : r”coldwave.|heatwave.|severe winter conditions.”, 

‘Storm’ : r”derecho.|hail.|lightning.|winterstorm.|storm surge.|tornado.|winter storm.|extra-tropical storm.|tropical storm.”
}


In [20]:
# select reports to be labelled
appealCode_luca = ["MDRLA009",
"MDRMG020",
"MDRNI012",
"MDRBZ006",
"MDRCN006",
"MDRBD022",
"MDRYE011",
"MDRS2001",
"MDRIQ014",
"MDRGN015",
"MDRSV012",
"MDRMY003",
"MDRBD015"]

reports_to_label_luca = filtered_reports.where(filtered_reports.appealCode.isin(appealCode_luca)).dropna()

#select most recent reports
reports_to_label_luca = reports_to_label_luca.groupby('appealCode').apply(lambda x: x.sort_values('date', ascending=False).head(1))

#check that everything is there
print(f"Number appealCodes: {len(appealCode_luca)}, number reports: {len(reports_to_label_luca)}")

Number appealCodes: 13, number reports: 13


In [32]:
filtered_reports.appealType.unique()

array(['DREF Operation', 'DREF Operation Final Report',
       'Operations Update', 'DREF Operation Update'], dtype=object)

In [22]:
reports_to_label_luca

,,reportName,disasterType,date,reportLink,location,appealCode,appealType,origType,pdfDownloaded,text,disasterTypeReclassified,disasterTypeFlag,naturalHazard,text_processed,sentences,nathaz_text
appealCode,,,,,,,,,,,,,,,,,
MDRBD015,1777,Bangladesh - Cyclone Komen (MDRBD015),Cyclone,16/09/2015,https://adore.ifrc.org/Download.aspx?FileId=97842,Bangladesh,MDRBD015,Operations Update,Operation Update no. 1,1.0,1 | P a g e \n \nSafe drinking water distribu...,Cyclone,0.0,1.0,1 | P a g e Safe drinking water distribution t...,[1 | P a g e Safe drinking water distribution ...,"[Meanwhile, Government district level ‘D-form’..."
MDRBD022,1029,Bangladesh - Monsoon Floods (MDRBD022),Flood,26/11/2019,https://adore.ifrc.org/Download.aspx?FileId=27...,Bangladesh,MDRBD022,Operations Update,Operations Update 2,1.0,\nEmergency Appeal n° MDRBD022 \nGLIDE n° FL-...,Flood,0.0,1.0,Emergency Appeal n° MDRBD022 GLIDE n° FL-2019-...,[Emergency Appeal n° MDRBD022 GLIDE n° FL-2019...,[The disaster risk reduction (DRR) activities ...
MDRBZ006,744,Belize - Hurricane Eta (MDRBZ006),All other disaster and emergencies,25/03/2021,https://adore.ifrc.org/Download.aspx?FileId=39...,Belize,MDRBZ006,DREF Operation Update,DREF Operation Update 1,1.0,\nInternal \n \nDREF Operation n° MDRBZ006 \n...,Cyclone,0.0,1.0,Internal DREF Operation n° MDRBZ006 GLIDE n° T...,[Internal DREF Operation n° MDRBZ006 GLIDE n° ...,[SITUATION ANALYSIS Description of the disaste...
MDRCN006,1265,China - Floods (MDRCN006),Flood,17/07/2018,https://adore.ifrc.org/Download.aspx?FileId=20...,China,MDRCN006,DREF Operation,DREF Operation,1.0,P a g e | 1 \n \n \n \nDREF n° MDRCN006 \nGli...,Flood,0.0,1.0,P a g e | 1 DREF n° MDRCN006 Glide n° TC-2018-...,[P a g e | 1 DREF n° MDRCN006 Glide n° TC-2018...,[P a g e | 1 DREF n° MDRCN006 Glide n° TC-2018...
MDRGN015,254,Guinea - Floods : Coyah (MDRGN015),Flood,31/08/2023,https://adore.ifrc.org/Download.aspx?FileId=72...,Guinea,MDRGN015,DREF Operation,MDRGN015do,1.0,DREF OPERATION\nGuinea Floods\nVolunteers cond...,Flood,0.0,1.0,DREF OPERATION Guinea Floods Volunteers conduc...,[DREF OPERATION Guinea Floods Volunteers condu...,[Appeal: MDRGN015 Country: Guinea Hazard: Floo...
MDRIQ014,589,Iraq - Flash Floods (MDRIQ014),Pluvial/Flash Flood,27/12/2021,https://adore.ifrc.org/Download.aspx?FileId=48...,Iraq,MDRIQ014,DREF Operation,DREF Operation,1.0,P a g e | 1 \n \n \n \n \n \nInternal \n \nDR...,Flood,0.0,1.0,P a g e | 1 Internal DREF Operation n° MDRIQ01...,[P a g e | 1 Internal DREF Operation n° MDRIQ0...,[P a g e | 1 Internal DREF Operation n° MDRIQ0...
MDRLA009,69,Laos - Flood (MDRLA009),Flood,31/05/2024,https://adore.ifrc.org/Download.aspx?FileId=83...,Lao People'S Democratic Republic,MDRLA009,DREF Operation Final Report,MDRLA009fnr,1.0,DREF Final Report\nDREF Laos Flood 2023\nAffec...,Flood,0.0,1.0,DREF Final Report DREF Laos Flood 2023 Affecte...,[DREF Final Report DREF Laos Flood 2023 Affect...,"[(Map: IFRC, IM) Date when the trigger was met..."
MDRMG020,350,Madagascar - Tropical Cyclone Freddy (MDRMG020),Cyclone,23/03/2023,https://adore.ifrc.org/Download.aspx?FileId=65...,Madagascar,MDRMG020,DREF Operation Update,MDRMG020du2,1.0,OPERATIONAL UPDATE\nMadagascar Tropical Cyclon...,Cyclone,0.0,1.0,OPERATIONAL UPDATE Madagascar Tropical Cyclone...,[OPERATIONAL UPDATE Madagascar Tropical Cyclon...,[OPERATIONAL UPDATE Madagascar Tropical Cyclon...
MDRMY003,1384,Malaysia - Floods (MDRMY003),Flood,21/11/2017,https://adore.ifrc.org/Download.aspx?FileId=17...,Malaysia,MDRMY003,DREF Operation Final Report,MDRMY003DREF_FR,1.0,\n \n \nDREF operation n° MDRMY003 \nGlide n°...,Flood,0.0,1.0,DREF operation n° MDRMY003 Glide n° FL-2017-00...,[DREF operation n° MDRMY003 Glide n° FL-2017-0...,[Situation analysis Description of the disaste...


In [40]:
#empty dict structure to store results
dict_=[
    {"hazardType": None,
     "hazardSubtypes" : None,
     "country" : None,
     "region" : None,
     "city" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },

]

labelled_reports_dict = {} # dict to store labelled reports

In [ ]:
i=1
print(reports_to_label_luca.iloc[i].appealCode)
reports_to_label_luca.iloc[i].nathaz_text

MDRBD015


['Meanwhile, Government district level ‘D-form’ data immediately after the disaster indicated many houses were flattened or went under water, trees uprooted, power supplies were disrupted, and communication systems ceased to operate in some places.',
 'Crops were damaged and shrimp projects flooded.',
 'Due to the impact of the cyclonic storm “Komen”, heavy to very heavy rainfall was active all over the country and many areas of the Emergency appeal operations update Bangladesh: Cyclone Komen 2 | P a g e southern Bangladesh were inundated which includes most of the areas affected by the first spell of flooding.',
 'Consequently the lives and livelihoods of the people of those areas further worsened.',
 'A Need Assessment Working Group (NAWG) was formed to identify the damage and needs of all these areas affected by the Cyclone Komen and subsequent flooding.',
 'This assessment was commissioned by the Humanitarian Coordination Task Team (HCTT) and was covered ten districts.',
 'The cumu

In [ ]:
labelled_reports_dict['MDRBD015']=[
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Bangladesh",
     "region" : "North and Central part",
     "city" : None,
     "locationAnnotation" : 'While BDRCS and IFRC as well as the other humanitarian partners are dealing with the cyclone Komen and flooding in the South Eastern part of Bangladesh, the North and Central part of Bangladesh is experiencing flooding since the last week of August 2015.',
     "startYear" : 2015,
     "startMonth" : 8,
     "startDay" : 20,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Bangladesh",
     "region" : "Jamuna and Brahmaputra River basin",
     "city" : None,
     "locationAnnotation" : 'The country is experiencing heavy to very heavy rainfall in the Jamuna and Brahmaputra River basin since last week of August.',
     "startYear" : 2015,
     "startMonth" : 8,
     "startDay" : 20,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Bangladesh",
     "region" : "Assam, Meghalaya, West Bengal",
     "city" : None,
     "locationAnnotation" : 'At the same time the upper catchment area of India, namely Assam, Meghalaya and some part of West Bengal also experienced heavy rain.',
     "startYear" : 2015,
     "startMonth" : 8,
     "startDay" : 20,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : ["tropical storm"],
     "country" : "Bangladesh",
     "region" : "South Eastern part",
     "city" : None,
     "locationAnnotation" : 'While BDRCS and IFRC as well as the other humanitarian partners are dealing with the cyclone Komen and flooding in the South Eastern part of Bangladesh, the North and Central part of Bangladesh is experiencing flooding since the last week of August 2015.',
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Komen",
    },
]

In [43]:
i=2
print(reports_to_label_luca.iloc[i].appealCode)
reports_to_label_luca.iloc[i].nathaz_text

MDRBZ006


['SITUATION ANALYSIS Description of the disaster Hurricane Eta made landfall on Nicaragua’s shores as a strong Category 4 hurricane on 4 November 2020, causing destruction and excessive rain with a wind speed of 140 mph.',
 'Several Central American countries experienced the negative effects of Hurricane Eta, including Belize.',
 'The rains started in Belize on 3 November 2020, increasing in intensity over the 4th and 5th of November.',
 'Approximately twenty inches of rainfall, caused severe flooding in the Western District of Cayo and Belize District, including Belize City.',
 'More than 40 communities were affected, mainly along the Mopan, Macal, Belize, and Sibun rivers.',
 'In the Cayo District, the Macal and Mopan rivers rose more than 8.8 meters, inundating every village from Arenal to Roaring Creek.',
 'Collective Centers were activated on 3 November 2020, to facilitate people living in swampy and low-lying areas.',
 'According to the National Emergency Management Agency (NEMO)

In [ ]:
labelled_reports_dict['MDRBZ006']=[
    {"hazardType": 'Storm',
     "hazardSubtypes" : ["tropical storm"],
     "country" : 'Nicaragua',
     "region" : None,
     "city" : None,
     "locationAnnotation" : 'SITUATION ANALYSIS Description of the disaster Hurricane Eta made landfall on Nicaragua’s shores as a strong Category 4 hurricane on 4 November 2020, causing destruction and excessive rain with a wind speed of 140 mph.',
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 4,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : 'Eta',
    },
    {"hazardType": 'Storm',
     "hazardSubtypes" : ["tropical storm"],
     "country" : 'Belize',
     "region" : None,
     "city" : None,
     "locationAnnotation" : 'Several Central American countries experienced the negative effects of Hurricane Eta, including Belize.',
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 3,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : 'Eta',
    },
    {"hazardType": 'Flood',
     "hazardSubtypes" : None,
     "country" : 'Belize',
     "region" : "Western District of Cayo",
     "city" : None,
     "locationAnnotation" :  'Approximately twenty inches of rainfall, caused severe flooding in the Western District of Cayo and Belize District, including Belize City.',
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 3,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": 'Flood',
     "hazardSubtypes" : None,
     "country" : 'Belize',
     "region" : "Belize District",
     "city" : None,
     "locationAnnotation" :  'Approximately twenty inches of rainfall, caused severe flooding in the Western District of Cayo and Belize District, including Belize City.',
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 3,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": 'Flood',
     "hazardSubtypes" : None,
     "country" : 'Belize',
     "region" : "Belize District",
     "city" : "Belize City",
     "locationAnnotation" :  'Approximately twenty inches of rainfall, caused severe flooding in the Western District of Cayo and Belize District, including Belize City.',
     "startYear" : 2020,
     "startMonth" : 11,
     "startDay" : 3,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
]

In [ ]:
i = 9
for report in filtered_reports :
    if report['appealCode']==appealCode_to_label[i] :
        print(len(report['header']))
        for sent in report['header'] :
            print(sent)
if appealCode_to_label[i] in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_to_label[i]])

11
SITUATION ANALYSIS Description of the crisis Mozambique is currently experiencing severe effects from the strong 20232024 El Nio season which brought below average rainfall to southern and central Mozambique and aboveaverage rainfall to the northern regions, severely impacting agriculture and rural livelihoods.
Additionally, Tropical Storm Filipo in March 2024 impacted 153,000 people, caused significant infrastructural damage, and further devastated agricultural lands, particularly in regions still reeling from the extensive destruction caused by TC Freddy in 2023 OCHA.
The compounded effects of these events have severely strained access to basic services and hindered recovery efforts OCHA.
Provinces such as Tete, Gaza, Manica, and Inhambane, known for high production and pastoral activities, have seen significant reductions in agricultural output with well belowaverage harvests compared to last year and the fiveyear average.
As of April 2024, approximately 690,000 hectares of crops

In [75]:
df_combined_labelled.to_csv(file_path_save+'labelled_example_haz-subtype-emdat_14-02.csv', index=False)

In [ ]:
dict_=[
    {"hazardType": "Storm",
     "hazardSubtypes" : "tropical storm",
     "country" : "Mozambique",
     "region" : None,
     "city" : None,
     "locationAnnotation" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Filippo",
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "tropical storm",
     "country" : "Mozambique",
     "region" : None,
     "city" : None,
     "locationAnnotation" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Freddy",
    },
    {"hazardType": "Drought",
     "hazardSubtypes" : "drought",
     "country" : "Mozambique",
     "region" : ["Tete", "Gaza", "Manica", "Inhambane", "southern", "central"],
     "city" : None,
     "locationAnnotation" : ["Provinces such as Tete, Gaza, Manica, and Inhambane, known for high production and pastoral activities, have seen significant reductions in agricultural output with well below average harvests compared to last year and the fiveyear average.",
                             "SITUATION ANALYSIS Description of the crisis Mozambique is currently experiencing severe effects from the strong 20232024 El Nio season which brought below average rainfall to southern and central Mozambique and above average rainfall to the northern regions, severely impacting agriculture and rural livelihoods."],
     "startYear" : 2024,
     "startMonth" : 4,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = appealCode_to_label[i]
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

if i==0 :
    df_combined_labelled = df
else :
    df_combined_labelled = pd.concat([df_combined_labelled, df], axis=0)

In [ ]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Zambia",
     "region" : ["Lusaka", "Luapula", "Western", "Southern", "Central", "Northwestern"],
     "city" : ["Mazabuka"],
     "locationAnnotation" : ["This includes more than three million children under 18 years of age, mostly based in the provinces of Lusaka, Luapula, and the Western, Southern, Central, and Northwestern Provinces.",
                             "Page 1 21 DREF Operation Zambia Drought 2024 Staff checking on a maize field affected by drought in Mazabuka District, Southern Province Appeal MDRZM022 Country Zambia Hazard Drought Type of DREF Response Crisis Category Orange Event Onset Slow DREF Allocation CHF 750,459 Glide Number People Affected 5,000,000 people People Targeted 160,000 people Operation Start Date 20240322 Operation Timeframe 6 months Operation End Date 30092024 DREF Published 28032024 Targeted Areas Southern Page 2 21 Description of the Event Date when the trigger was met 20240229 Districts affeccted by Drought What happened, where and when?",
                             "The provinces affected include NorthWestern, Southern, Western, Central and Eastern."
                            ],
     "startYear" : 2024,
     "startMonth" : 2,
     "startDay" : 29,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = appealCode_to_label[i]
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

if i==0 :
    df_combined_labelled = df
else :
    df_combined_labelled = pd.concat([df_combined_labelled, df], axis=0)

In [ ]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Nigeria",
     "region" : ["Bauchi", "Kebbi", "Sokoto", "Zamfara"],
     "city" : ["Kano", "Maiduguri", "Giade", "Shira", "Katagum"],
     "locationAnnotation" : ["DREF Operation Nigeria Floods DREF 2024 Flood cuts off major access road linking Kano to Maiduguri in Katagum community, Bauchi state Appeal MDRNG041 Country Nigeria Hazard Flood Type of DREF Response Crisis Category Yellow Event Onset Sudden DREF Allocation CHF 231,293 Glide Number People Affected 50,000 people People Targeted 9,000 people Operation Start Date 03092024 Operation Timeframe 4 months Operation End Date 31012025 DREF Published 06092024 Targeted Areas Bauchi, Kebbi, Sokoto, Zamfara Page 1 17 Description of the Event Date of event 13082024 Nigeria Flood Forecast 2024 What happened, where and when?",
                             "From August 8 to August 13, 2024, continuous heavy rainfall triggered severe flooding across Nigeria, leading to widespread devastation and displacement in states such as Bauchi, Sokoto, and Zamfara.",
                             "In Bauchi State, over 1,000 homes were destroyed, particularly impacting the Giade, Shira, and Katagum local government areas."
                            ],
     "startYear" : 2024,
     "startMonth" : 8,
     "startDay" : 8,
     "endYear" : 2024,
     "endMonth" : 8,
     "endDay" : 13,
     "hazardName" : None,
    },
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Nigeria",
     "region" : ["Sokoto"],
     "city" : ["Gada", "Dantudu", "Balakozo", "Gidan Tudu", "Tsitse"],
     "locationAnnotation" : ["Earlier, on July 17, 2024, flooding in Sokoto State displaced 1,664 people and caused extensive damage to farmlands and livestock across four communities in Gada Local Government Area, including Dantudu, Balakozo, Gidan Tudu, and Tsitse."
                            ],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = appealCode_to_label[i]
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

if i==0 :
    df_combined_labelled = df
else :
    df_combined_labelled = pd.concat([df_combined_labelled, df], axis=0)

In [ ]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : ["flash flood", "riverine flood"],
     "country" : "Sudan",
     "region" : ["Red Sea", "River Nile", "Northern State"],
     "city" : None,
     "locationAnnotation" : ["So far, Red Sea, River Nile, and Northern State have been the most severely affected."
                            ],
     "startYear" : 2024,
     "startMonth" : 6,
     "startDay" : 1,
     "endYear" : 2024,
     "endMonth" : 8,
     "endDay" : 12,
     "hazardName" : None,
    },
]
df = pd.DataFrame(dict_)
df['appealCode'] = appealCode_to_label[i]
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

if i==0 :
    df_combined_labelled = df
else :
    df_combined_labelled = pd.concat([df_combined_labelled, df], axis=0)

In [ ]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : ["flash flood", "riverine flood"],
     "country" : "Benin",
     "region" : ["Couffo", "Adoukandji", "Ahomadegbe", "Gnizounme", "Tchito", "Tohou", "Zalli"],
     "city" : ["Mono", "Couffo", "Zou", "Oum", "Ahouada", "Hazin", "Yamontou", "Ahomadegbe", "Gnizounme", "Hangbannou", "Tandji", "Aboti", "Zounhome", "Hehokpa", "Sawanou", "Tohou Centre", "Adjassagon"],
     "locationAnnotation" : ["Intense rainfall observed in the departments of Mono, Couffo, Zou and Oum in the South of Benin caused the overflow of the river Couffo on 26 June 2024 in 6 of the 11 districts of the commune it crosses in Couffo department, Adoukandji, Ahomadegbe, Gnizounme, Tchito, Tohou and Zalli.",
                             "A rapid assessment conducted during the following days by Benin Red Cross and the Lalo council on July 1, 2024 indicates that about 13 villages Ahouada, Hazin, Yamontou, Ahomadegbe, Gnizounme, Hangbannou, Tandji, Aboti, Zounhome, Hehokpa, Sawanou, Tohou Centre and Adjassagon were flooded with several houses destroyed and damaged.",
                            ],
     "startYear" : 2024,
     "startMonth" : 6,
     "startDay" : 26,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
]
df = pd.DataFrame(dict_)
df['appealCode'] = appealCode_to_label[i]
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

if i==0 :
    df_combined_labelled = df
else :
    df_combined_labelled = pd.concat([df_combined_labelled, df], axis=0)

In [ ]:
i=2
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : ["flash flood", "riverine flood"],
     "country" : "Cameroon",
     "region" : ["Cameroons Far North region", "Logone", "Chari", "Mayo Danay", "Diamar"],
     "city" : ["Blangoua", "Mackary", "Zina", "Maga", "Yagoua", "Ndoukoula", "Mokolo"],
     "locationAnnotation" : ["Series of floods have been recorded since August 19, reaching critical levels in the Logone et Chari and Mayo Danay divisions between August 11 and 21, 2024.",
                             "The most affected districts are Blangoua, Mackary, and Zina in the Logone and Chari department, and Maga, Yagoua in the Mayo Danay division.",
                             "In the Logone et Chari division, the affected districts are Blangoua with nearly 75,000 people affected Makary with 43,000 Zina with 9,000 people affected In the Mayo Danay division Maga with 18,000 people affected Yagoua with nearly 13,000 people The rains continue with weather forecasts predicting more significant impacts in the divisions already mentioned Page 2 19 above, as well as in others that have also been experiencing heavy rainfall for several days.",
                             "Notably, in the Diamar division, where Ndoukoula district has reported over 400 people affected to date, while in Mayo Tsanaga, Mokolo district, has recorded nearly 200 affected people."
                            ],
     "startYear" : 2024,
     "startMonth" : 8,
     "startDay" : 10,
     "endYear" : 2024,
     "endMonth" : 8,
     "endDay" : 28,
     "hazardName" : None,
    },
]
df = pd.DataFrame(dict_)
df['appealCode'] = appealCode_to_label[i]
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

if i==0 :
    df_combined_labelled = df
else :
    df_combined_labelled = pd.concat([df_combined_labelled, df], axis=0)
# df_combined_examples.to_csv(file_path_save+'labelled_example_haz-type-emdat.csv', index=False)

In [ ]:
i=1
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : ["flash flood", "riverine flood"],
     "country" : "Pakistan",
     "region" : ["Balochistan ", "Sindh ", "Punjab", "Khyber Pakhtunkhwa", "Azad Jammu"],
     "city" : ["Jacobabad", "Naushahro Feroz", "Ghotki", "Sukkur", "Sanghar", "Dadu", "Shaheed Benazirabad", "Kashmor", "Taluka Tando Adam"],
     "locationAnnotation" : ["Regionally, Balochistan received 239 per cent more rainfall than usual, Sindh 318 per cent, Punjab 111 per cent, and Khyber Pakhtunkhwa KP 25 per cent.",
                             "This unprecedented volume of rainfall, coupled with unusually high temperatures, accelerated snowmelt in KP, Azad Jammu and Kashmir AJK, and Gilgit Baltistan GB, leading to catastrophic flash floods and landslides.",
                             "Areas such as Jacobabad, Naushahro Feroz, Ghotki, Sukkur, Sanghar, Dadu, Shaheed Benazirabad, and Kashmor have faced heavy rains, resulting in substantial damage to homes and infrastructure.",
                             "In district Sanghar, the Deputy Commissioner reported a massive breach in the Rohri Canal that created numerous water bodies in Taluka Tando Adam, inundating over 35 villages and displacing 9,500 people who are now residing in relief camps."
                            ],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 9,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Mass movement",
     "hazardSubtypes" : "landslide",
     "country" : "Pakistan",
     "region" : ["Khyber Pakhtunkhwa", "Azad Jammu and Kashmir", "Gilgit Baltistan"],
     "city" : None,
     "locationAnnotation" : ["This unprecedented volume of rainfall, coupled with unusually high temperatures, accelerated snowmelt in KP, Azad Jammu and Kashmir AJK, and Gilgit Baltistan GB, leading to catastrophic flash floods and landslides"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Extreme temperature",
     "hazardSubtypes" : "heatwave",
     "country" : "Pakistan",
     "region" : ["Khyber Pakhtunkhwa", "Azad Jammu and Kashmir", "Gilgit Baltistan"],
     "city" : None,
     "locationAnnotation" : ["This unprecedented volume of rainfall, coupled with unusually high temperatures, accelerated snowmelt in KP, Azad Jammu and Kashmir AJK, and Gilgit Baltistan GB, leading to catastrophic flash floods and landslides."],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = appealCode_to_label[i]
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

if i==0 :
    df_combined_labelled = df
else :
    df_combined_labelled = pd.concat([df_combined_labelled, df], axis=0)
# df_combined_examples.to_csv(file_path_save+'labelled_example_haz-type-emdat.csv', index=False)

In [ ]:
i=0
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Algeria",
     "region" : "southern and western Algeria",
     "city" : ["Bchar", "Elbayadh", "Beni Abbes", "Tamanrasset", "Tiaret", "Tindouf", "Naama"],
     "locationAnnotation" : ["The most affected areas include Bchar, Elbayadh, Beni Abbes, Tamanrasset, Tiaret, Tindouf, and Naama."],
     "startYear" : 2024,
     "startMonth" : 9,
     "startDay" : 5,
     "endYear" : 2024,
     "endMonth" : 9,
     "endDay" : 8,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = appealCode_to_label[i]
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

if i==0 :
    df_combined_labelled = df
else :
    df_combined_labelled = pd.concat([df_combined_labelled, df], axis=0)
# df_combined_examples.to_csv(file_path_save+'labelled_example_haz-type-emdat.csv', index=False)

In [8]:
filtered_reports[1]['header']

['Map IFRC, IM What happened, where and when?',
 'Pakistan has experienced an unusually intense and prolonged monsoon season, resulting in widespread infrastructure damage, numerous casualties, and significant injuries.',
 'The season, which began in July 2024, continued through August, with particularly heavy rainfall recorded throughout the month.',
 'The latest significant weather spell, from 26 August to 1 September 2024, exacerbated the situation.',
 'During this period, the Pakistan Meteorological Department PMD issued forecasts of additional heavy rainfall, worsening the already critical conditions.',
 'The monsoon rains have been exceptionally severe, with rainfall levels reaching up to 318 per cent above normal in some areas.',
 'Regionally, Balochistan received 239 per cent more rainfall than usual, Sindh 318 per cent, Punjab 111 per cent, and Khyber Pakhtunkhwa KP 25 per cent.',
 'This unprecedented volume of rainfall, coupled with unusually high temperatures, accelerated sn

In [ ]:
dict_1 = [
    {
        "Hazard": "Flood",
        "Country": "Pakistan",
        "Locations": ["Balochistan ", "Sindh ", "Punjab", "Khyber Pakhtunkhwa KP", "Azad Jammu", "Khyber Pakhtunkhwa KP",
                     "Jacobabad, Naushahro Feroz, Ghotki, Sukkur, Sanghar, Dadu, Shaheed Benazirabad, and Kashmor",
                     "Taluka Tando Adam"],
        "Start_Date": "July 2024",
        "End_Date": "NULL",
    },
    {
        "Hazard": "Mass movement",
        "Country": "Pakistan",
        "Locations": ["KP, Azad Jammu and Kashmir AJK, and GilgitBaltistan GB"],
        "Start_Date": "NULL",
        "End_Date": "NULL",
    },
    {
        "Hazard": "Extreme temperature",
        "Country": "Pakistan",
        "Locations": "NULL",
        "Start_Date": "26 August 2024",
        "End_Date": "1 September 2024",
    }
]
df_1 = pd.DataFrame(dict_1)
df_1['appealCode'] = filtered_reports[1]['appealCode']
df_1['Country'] = [country_name_to_iso3(cntr) for cntr in df_1['Country']]

In [10]:
#filtered_reports[2]['appealCode']
filtered_reports[2]['header']

['Appeal MDRCM039 Country Cameroon Hazard Flood Type of DREF Response Crisis Category Yellow Event Onset Slow DREF Allocation CHF 421,471 Glide Number People Affected 158,620 people People Targeted 4,800 people Operation Start Date 04092024 Operation Timeframe 5 months Operation End Date 28022025 DREF Published 13092024 Targeted Areas ExtrmeNord Page 1 19 Description of the Event Date when the trigger was met 28082024 Carte de lExtrme Nord What happened, where and when?',
 'Cameroons Far North region has been experiencing flooding since the start of the rainy season, which began in the second half of July with an average rainfall frequency of one day out of four.',
 'The intensification and recurrence of rains starting from August 10, 2024, has led to a progressive increase in rainfall levels between August 10 and August 19, 2024.',
 'Series of floods have been recorded since August 19, reaching critical levels in the Logone et Chari and Mayo Danay divisions between August 11 and 21, 2

In [ ]:
dict_2 = [
    {
        "Hazard": "Flood",
        "Country": "Cameroon",
        "Locations": ["Cameroons Far North region", "Logone et Chari and Mayo Danay", "Yagoua", "Blangoua, Mackary, and Zina",
                      "Chari division", "Maga, Yagoua", "Logone division", "Ndoukoula district"],
        "Start_Date": "second half of July 2024",
        "End_Date": "August 28, 2024",
    },
    {
        "Hazard": "Drought",
        "Country": "Cameroon",
        "Locations": "NULL",
        "Start_Date": "2024",
        "End_Date": "NULL",
    }
]

df_2 = pd.DataFrame(dict_2)
df_2['appealCode'] = filtered_reports[2]['appealCode']
df_2['Country'] = [country_name_to_iso3(cntr) for cntr in df_2['Country']]

In [12]:
#filtered_reports[3]['appealCode']
filtered_reports[3]['header']

['DREF Operation BeninFlood in Lalo Field visits to displaced communities hosted in a school in Couffo RCB Appeal MDRBJ019 Country Benin Hazard Flood Type of DREF Response Crisis Category Yellow Event Onset Sudden DREF Allocation CHF 254,682 Glide Number People Affected 34,052 people People Targeted 10,215 people Operation Start Date 12072024 Operation Timeframe 4 months Operation End Date 30112024 DREF Published 23072024 Targeted Areas Couffo Page 1 17 Description of the Event Date of event 26062024 MAP most affected district by Red Cross of Benin What happened, where and when?',
 'Intense rainfall observed in the departments of Mono, Couffo, Zou and Oum in the South of Benin caused the overflow of the river Couffo on 26 June 2024 in 6 of the 11 districts of the commune it crosses in Couffo department, Adoukandji, Ahomadegbe, Gnizounme, Tchito, Tohou and Zalli.',
 'A rapid assessment conducted during the following days by Benin Red Cross and the Lalo council on July 1, 2024 indicates 

In [ ]:
dict_3 = [
    {
        "Hazard": "Flood",
        "Country": "Benin",
        "Locations": ["Mono, Couffo, Zou and Oum in the South of Benin", "Couffo department, Adoukandji, Ahomadegbe, Gnizounme, Tchito, Tohou and Zalli",
                      "Ahouada, Hazin, Yamontou, Ahomadegbe, Gnizounme, Hangbannou, Tandji, Aboti, Zounhome, Hehokpa, Sawanou, Tohou Centre and Adjassagon"
                     ],
        "Start_Date": "26 June 2024",
        "End_Date": "NULL",
    }
]

df_3 = pd.DataFrame(dict_3)
df_3['appealCode'] = filtered_reports[3]['appealCode']
df_3['Country'] = [country_name_to_iso3(cntr) for cntr in df_3['Country']]

In [14]:
#print(filtered_reports[4]['appealCode'])
print(filtered_reports[4]['header'])

['IFRC, IM What happened, where and when?', 'Sudan has been grappling with heavy rains that have led to widespread flooding across many regions, worsening the already dire situation caused by the conflict that began 16 months ago.', 'Between June 1 and August 12, 2024, DTM Sudan reported 60 incidents of heavy rains and floods, resulting in sudden displacement IOM.', 'The rainy season is expected to continue until October 2024, with forecasts predicting aboveaverage rainfall.', 'The likelihood of additional flash floods and river flooding remains high.', 'So far, Red Sea, River Nile, and Northern State have been the most severely affected.', 'Sudan has faced flooding in recent years, but the current humanitarian crisis, combined with heavy rains and deadly floods, has had a devastating impact on communities.', 'Both those displaced by conflict and the host communities supporting them under already challenging conditions are suffering.', 'The floods have rendered roads impassable, furthe

In [ ]:
dict_4 = [
    {
        "Hazard": "Flood",
        "Country": "Sudan",
        "Locations": ["Red Sea, River Nile, and Northern State"
                     ],
        "Start_Date": "1 June 2024",
        "End_Date": "12 August 2024",
    }
]
df_4 = pd.DataFrame(dict_4)
df_4['appealCode'] = filtered_reports[4]['appealCode']
df_4['Country'] = [country_name_to_iso3(cntr) for cntr in df_4['Country']]

In [16]:
#print(filtered_reports[5]['appealCode'])
print(filtered_reports[5]['header'])

['DREF Operation Nigeria Floods DREF 2024 Flood cuts off major access road linking Kano to Maiduguri in Katagum community, Bauchi state Appeal MDRNG041 Country Nigeria Hazard Flood Type of DREF Response Crisis Category Yellow Event Onset Sudden DREF Allocation CHF 231,293 Glide Number People Affected 50,000 people People Targeted 9,000 people Operation Start Date 03092024 Operation Timeframe 4 months Operation End Date 31012025 DREF Published 06092024 Targeted Areas Bauchi, Kebbi, Sokoto, Zamfara Page 1 17 Description of the Event Date of event 13082024 Nigeria Flood Forecast 2024 What happened, where and when?', 'From August 8 to August 13, 2024, continuous heavy rainfall triggered severe flooding across Nigeria, leading to widespread devastation and displacement in states such as Bauchi, Sokoto, and Zamfara.', 'In Bauchi State, over 1,000 homes were destroyed, particularly impacting the Giade, Shira, and Katagum local government areas.', 'Earlier, on July 17, 2024, flooding in Sokoto

In [ ]:
dict_5 = [
    {
        "Hazard": "Flood",
        "Country": "Nigeria",
        "Locations": ["Kano", "Maiduguri", "Bauchi state", "Bauchi, Kebbi, Sokoto, Zamfara"
                     ],
        "Start_Date": "8 August 2024",
        "End_Date": "13 August 2024",
    },
    {
        "Hazard": "Flood",
        "Country": "Nigeria",
        "Locations": ["Sokoto State", "Dantudu, Balakozo, Gidan Tudu, and Tsitse", "Zamfara State", "Ruwan Gora, Morai, Makera, and Talata Mafara town"],
        "Start_Date": "17 July 2024",
        "End_Date": "NULL",
    }
]
df_5 = pd.DataFrame(dict_5)
df_5['appealCode'] = filtered_reports[5]['appealCode']
df_5['Country'] = [country_name_to_iso3(cntr) for cntr in df_5['Country']]

In [18]:
#print(filtered_reports[6]['appealCode'])
print(filtered_reports[6]['header'])

['Page 1 23 Description of the Event Map of the areas most affected by the disaster Date of event 06052023 What happened, where and when?', 'From 1 to 6 May, Rwanda experienced continuous torrential rains, which caused major damage in several Districts of the country.', 'According to assessments carried out by the Rwanda Red Cross and other stake holders and MINEMA led, the western, northern and southern provinces of Rwanda were the areas hardest hit by the flooding.', 'Overall, 14 districts experienced flooding and landslides affecting around 51,905 people in 10,381 households.', 'A total of 137 people died, and 5,472 houses were destroyed.', 'Damage reported includes major losses of houses, basic household items, unusable water sources, latrines and roads.', 'The destruction of thousands of hectares of crops and livestock was immense.', 'Those affected were gathered together in IDP sites.', 'The needs were enormous and the vulnerabilities high.', 'The rains continued until June 2023.

In [ ]:
dict_6 = [
    {
        "Hazard": "Flood",
        "Country": "Rwanda",
        "Locations": ["western, northern and southern provinces", "14 districts"
                     ],
        "Start_Date": "1 May 2023",
        "End_Date": "June 2023",
    },
    {
        "Hazard": "Mass movement",
        "Country": "Rwanda",
        "Locations": ["14 districts"
                     ],
        "Start_Date": "1 May 2023",
        "End_Date": "NULL",
    },
]
df_6 = pd.DataFrame(dict_6)
df_6['appealCode'] = filtered_reports[6]['appealCode']
df_6['Country'] = [country_name_to_iso3(cntr) for cntr in df_6['Country']]

In [20]:
#print(filtered_reports[7]['appealCode'])
print(filtered_reports[7]['header'])

['SITUATION ANALYSIS Description of the crisis Zambia is undergoing one of the driest agricultural seasons in more than forty years, causing major crop and livestock losses and severely affecting the wellbeing and livelihoods of communities nationwide.', 'According to ongoing reports from the UN, 84 out of 116 districts in the country have been affected by this crisis.', 'The IPC report from August 20231 projected an estimated 58,000 people, between October 2023 and March 2024, to be in an Emergency condition IPC Phase 4 and two million people in Crisis IPC Phase 3 and requiring urgent humanitarian support.', 'On 29 February 2024, the President of Zambia declared a national emergency due to the prolonged drought.', 'On 16 April 2024, the joint rapid needs assessment 2 was commissioned by the Agriculture and Food Security Cluster and the National Government Drought Response Appeal indicated that 6.6 million people needed urgent humanitarian assistance 33 per cent of Zambias total popula

In [21]:
dict_7 = [
    {
        "Hazard": "Drought",
        "Country": "Zambia",
        "Locations": ["Lusaka, Luapula, and the Western, Southern, Central, and Northwestern Provinces", "Western, Southern, and NorthWestern."
                     ],
        "Start_Date": "29 February 2024",
        "End_Date": "NULL",
    }
]
df_7 = pd.DataFrame(dict_7)
df_7['appealCode'] = filtered_reports[7]['appealCode']
df_7['Country'] = [country_name_to_iso3(cntr) for cntr in df_7['Country']]

In [22]:
#print(filtered_reports[8]['appealCode'])
print(filtered_reports[8]['header'])

['Appeal MDRUG050 Total DREF Allocation CHF 479,715 Crisis Category Yellow Hazard Flood Glide Number People Affected 69,283 people People Targeted 19,098 people Event Onset Slow Operation Start Date 22052024 New Operational End Date 30112024 Total Operating Timeframe 6 months Reporting Timeframe Start Date Reporting Timeframe End Date Additional Allocation Requested 157,941 Targeted Areas Central Region, Eastern Region, Western Region Page 1 19 Description of the Event Map of Uganda showing flood affected districts Date when the trigger was met 21082024 What happened, where and when?', 'In April 2024, the Eastern UgandaElgon region experienced heavy rainfall, as forecasted by the Uganda National Meteorological Authority UNMA.', 'This resulted in significant impacts from episodic floods, hailstorms, and landslides in various areas, including Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa.', 'A total of 18,323 people were affected, including thousands of 

In [ ]:
dict_8 = [
    {
        "Hazard": "Flood",
        "Country": "Uganda",
        "Locations": ["Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa",
                      "Manafwa, Lwakhakha, Sironko, Mpologoma, Awoja, Nbuyonga, and Namatala"],
        "Start_Date": "April 2024",
        "End_Date": "31st August 2024",
    },
    {
        "Hazard": "Storm",
        "Country": "Uganda",
        "Locations": ["Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa"],
        "Start_Date": "NULL",
        "End_Date": "NULL",
    },
    {
        "Hazard": "Mass movement",
        "Country": "Uganda",
        "Locations": ["Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa"],
        "Start_Date": "NULL",
        "End_Date": "NULL",
    }
]
df_8 = pd.DataFrame(dict_8)
df_8['appealCode'] = filtered_reports[8]['appealCode']
df_8['Country'] = [country_name_to_iso3(cntr) for cntr in df_8['Country']]

In [24]:
#print(filtered_reports[9]['appealCode'])
print(filtered_reports[9]['header'])

['SITUATION ANALYSIS Description of the crisis Mozambique is currently experiencing severe effects from the strong 20232024 El Nio season which brought below average rainfall to southern and central Mozambique and aboveaverage rainfall to the northern regions, severely impacting agriculture and rural livelihoods.', 'Additionally, Tropical Storm Filipo in March 2024 impacted 153,000 people, caused significant infrastructural damage, and further devastated agricultural lands, particularly in regions still reeling from the extensive destruction caused by TC Freddy in 2023 OCHA.', 'The compounded effects of these events have severely strained access to basic services and hindered recovery efforts OCHA.', 'Provinces such as Tete, Gaza, Manica, and Inhambane, known for high production and pastoral activities, have seen significant reductions in agricultural output with well belowaverage harvests compared to last year and the fiveyear average.', 'As of April 2024, approximately 690,000 hectar

In [ ]:
dict_9 = [
    {
        "Hazard": "Storm", #Freddy
        "Country": "Mozambique",
        "Locations": [],
        "Start_Date": "March 2024",
        "End_Date": "NULL",
    },
    {
        "Hazard": "Storm", #Filippo
        "Country": "Mozambique",
        "Locations": [],
        "Start_Date": "2023",
        "End_Date": "NULL",
    },
    {
        "Hazard": "Drought",
        "Country": "Mozambique",
        "Locations": ["central and northern zones"],
        "Start_Date": "May 2024",
        "End_Date": "June 2024",
    }
]
df_9 = pd.DataFrame(dict_9)
df_9['appealCode'] = filtered_reports[9]['appealCode']
df_9['Country'] = [country_name_to_iso3(cntr) for cntr in df_9['Country']]

In [27]:
df_combined_examples = pd.concat([df_0, df_1, df_2, df_3, df_4, df_5, df_6, df_7, df_8, df_9], axis=0)
df_combined_examples.to_csv(file_path_save+'labelled_example_haz-type-emdat.csv', index=False)